In [8]:
import pandas as pd
import numpy as np
import lightgbm as lgb

# ── 1. Load PFR season defense stats (training data) ────────────────────────
pfr = pd.read_csv(
    'https://github.com/nflverse/nflverse-data/releases/download/pfr_advstats/advstats_season_def.csv',
    low_memory=False
)

cb_pfr = pfr[
    (pfr['pos'] == 'CB') &
    (pfr['season'].between(2020, 2024)) &
    (pfr['tgt'] > 0)
].copy()

cb_pfr['incompletions']     = cb_pfr['tgt'] - cb_pfr['cmp'] - cb_pfr['int']
cb_pfr['incompletion_rate'] = cb_pfr['incompletions'] / cb_pfr['tgt']
cb_pfr['int_rate']          = cb_pfr['int']            / cb_pfr['tgt']
cb_pfr['td_rate_allowed']   = cb_pfr['td']             / cb_pfr['tgt']

# ── 2. Load snap counts (training) ───────────────────────────────────────────
snap_dfs = []
for season in [2020, 2021, 2022, 2023, 2024]:
    print(f"Loading snaps {season}...")
    df = pd.read_csv(
        f'https://github.com/nflverse/nflverse-data/releases/download/snap_counts/snap_counts_{season}.csv.gz',
        compression='gzip', low_memory=False
    )
    df['season'] = season
    snap_dfs.append(df)

snaps = pd.concat(snap_dfs, ignore_index=True)

cb_snaps = (
    snaps[snaps['position'] == 'CB']
    .groupby(['pfr_player_id', 'player'])
    .agg(career_snaps=('defense_snaps', 'sum'))
    .reset_index()
)

qualified = cb_snaps[cb_snaps['career_snaps'] >= 2500][['pfr_player_id', 'player', 'career_snaps']]

# ── 3. Aggregate and compute features ────────────────────────────────────────
cb_agg = cb_pfr.groupby(['player', 'pfr_id']).agg(
    total_targets     = ('tgt',           'sum'),
    total_incomp      = ('incompletions', 'sum'),
    total_int         = ('int',           'sum'),
    total_td          = ('td',            'sum'),
    avg_passer_rating = ('rat',           'mean'),
).reset_index()

cb_agg['incompletion_rate'] = cb_agg['total_incomp'] / cb_agg['total_targets']
cb_agg['int_rate']          = cb_agg['total_int']    / cb_agg['total_targets']
cb_agg['td_rate_allowed']   = cb_agg['total_td']     / cb_agg['total_targets']
cb_agg['passer_rating_inv'] = 158.3 - cb_agg['avg_passer_rating']

cb_agg = cb_agg.merge(qualified, left_on='pfr_id', right_on='pfr_player_id', how='inner')
cb_agg['target_rate']     = cb_agg['total_targets'] / cb_agg['career_snaps']
cb_agg['target_rate_inv'] = 1 - cb_agg['target_rate']

# ── 4. Assign labels ─────────────────────────────────────────────────────────
rankings = {
    'Marshon Lattimore':     1.00,
    'Jaire Alexander':       0.95,
    'Darius Slay':           0.90,
    'Patrick Surtain':       0.85,
    'Ahmad Gardner':         0.85,
    'Denzel Ward':           0.80,
    'Trent McDuffie':        0.80,
    'Stephon Gilmore':       0.75,
    'Charvarius Ward':       0.75,
    'LJarius Sneed':         0.75,
    'Jalen Ramsey':          0.70,
    'Marlon Humphrey':       0.70,
    'DJ Reed':               0.70,
    'Trevon Diggs':          0.70,
    'Xavien Howard':         0.65,
    'James Bradberry':       0.65,
    'Patrick Peterson':      0.60,
    'Adoree Jackson':        0.60,
    'TreDavious White':      0.60,
    'Tariq Woolen':          0.55,
    'Carlton Davis':         0.55,
    'Greg Newsome':          0.55,
    'AJ Terrell':            0.50,
    'Paulson Adebo':         0.50,
    'Kristian Fulton':       0.50,
    'Chidobe Awuzie':        0.45,
    'Levi Wallace':          0.45,
    'Darious Williams':      0.40,
    'Byron Murphy':          0.40,
    'Cameron Sutton':        0.40,
    'Marco Wilson':          0.40,
    'Dane Jackson':          0.40,
    'Donte Jackson':         0.40,
    'Benjamin St-Juste':     0.35,
    'Michael Davis':         0.35,
    'Taron Johnson':         0.35,
    'Alontae Taylor':        0.35,
    'Kendall Fuller':        0.35,
    'Sean Murphy-Bunting':   0.35,
    'Steven Nelson':         0.35,
    'Shaquill Griffin':      0.30,
    'Keisean Nixon':         0.30,
    'Mike Hilton':           0.30,
    'Jourdan Lewis':         0.30,
    'Fabian Moreau':         0.30,
    'Nate Hobbs':            0.25,
    'Troy Hill':             0.25,
    'Rasul Douglas':         0.25,
    'Kader Kohou':           0.20,
    'Kenny Moore':           0.20,
    'Amani Oruwariye':       0.20,
    'Asante Samuel':         0.20,
    'Ahkello Witherspoon':   0.20,
    'Tyson Campbell':        0.15,
    'Eli Apple':             0.15,
    'Michael Jackson':       0.15,
    'Jamel Dean':            0.15,
    'Ronald Darby':          0.10,
    'Desmond King':          0.10,
    'Chandon Sullivan':      0.10,
}

cb_agg['label'] = cb_agg['player_x'].map(rankings)
labeled = cb_agg[
    (cb_agg['label'].notna()) &
    (cb_agg['total_targets'] >= 50)
].copy()

labeled['relevance'] = (labeled['label'] * 4).round().astype(int)

# ── 5. Train LambdaRank ───────────────────────────────────────────────────────
features = ['incompletion_rate', 'int_rate', 'passer_rating_inv', 'target_rate_inv']

X = labeled[features].values
y = labeled['relevance'].values
group = [len(X)]

train_data = lgb.Dataset(X, label=y, group=group)

params = {
    'objective':        'lambdarank',
    'metric':           'ndcg',
    'ndcg_eval_at':     [10],
    'learning_rate':    0.05,
    'num_leaves':       8,
    'min_data_in_leaf': 1,
    'verbose':          -1,
}

model = lgb.train(params, train_data, num_boost_round=200)
print("Model trained successfully")

# ── 6. Load 2019 weekly data and aggregate ───────────────────────────────────
print("\nLoading 2019 data...")
pfr_2019 = pd.read_csv(
    'https://github.com/nflverse/nflverse-data/releases/download/pfr_advstats/advstats_week_def_2019.csv',
    low_memory=False
)

pfr_2019_season = pfr_2019[pfr_2019['game_type'] == 'REG'].groupby(
    ['pfr_player_name', 'pfr_player_id']
).agg(
    tgt     = ('def_targets',              'sum'),
    cmp     = ('def_completions_allowed',  'sum'),
    int_    = ('def_ints',                 'sum'),
    td      = ('def_receiving_td_allowed', 'sum'),
    avg_rat = ('def_passer_rating_allowed', 'mean'),
).reset_index()

# ── 7. Load 2019 snap counts ─────────────────────────────────────────────────
snaps_2019 = pd.read_csv(
    'https://github.com/nflverse/nflverse-data/releases/download/snap_counts/snap_counts_2019.csv.gz',
    compression='gzip', low_memory=False
)

cb_snaps_2019 = (
    snaps_2019[snaps_2019['position'] == 'CB']
    .groupby(['pfr_player_id', 'player'])
    .agg(season_snaps=('defense_snaps', 'sum'))
    .reset_index()
)

# ── 8. Merge and compute features ────────────────────────────────────────────
cb_2019 = pfr_2019_season.merge(
    cb_snaps_2019[cb_snaps_2019['season_snaps'] >= 500],
    on='pfr_player_id',
    how='inner'
)

cb_2019 = cb_2019[cb_2019['tgt'] >= 20].copy()

cb_2019['incompletions']     = cb_2019['tgt'] - cb_2019['cmp'] - cb_2019['int_']
cb_2019['incompletion_rate'] = cb_2019['incompletions']  / cb_2019['tgt']
cb_2019['int_rate']          = cb_2019['int_']           / cb_2019['tgt']
cb_2019['passer_rating_inv'] = 158.3 - cb_2019['avg_rat']
cb_2019['target_rate']       = cb_2019['tgt']            / cb_2019['season_snaps']
cb_2019['target_rate_inv']   = 1 - cb_2019['target_rate']

# ── 9. Score and rank 2019 CBs ───────────────────────────────────────────────
X_2019 = cb_2019[features].values
cb_2019['wcs'] = model.predict(X_2019)

cb_2019_ranked = cb_2019[['player', 'tgt', 'incompletion_rate', 'int_rate', 'target_rate', 'avg_rat', 'wcs']]\
    .sort_values('wcs', ascending=False)\
    .reset_index(drop=True)

cb_2019_ranked.index += 1

print(f"\n2019 CB Rankings ({len(cb_2019_ranked)} CBs):")
print(cb_2019_ranked.to_string())

Loading snaps 2020...
Loading snaps 2021...
Loading snaps 2022...
Loading snaps 2023...
Loading snaps 2024...
Model trained successfully

Loading 2019 data...

2019 CB Rankings (94 CBs):
                   player  tgt  incompletion_rate  int_rate  target_rate     avg_rat       wcs
1        Bashaud Breeland   62           0.483871  0.032258     0.055806   84.886667  1.222676
2          Trayvon Mullen   68           0.426471  0.014706     0.100741   80.041667 -0.993592
3     William Jackson III   67           0.402985  0.014925     0.080626   81.571429 -1.128623
4               Eric Rowe   71           0.408451  0.014085     0.066293   74.193750 -1.269535
5           Steven Nelson   74           0.486486  0.013514     0.073195   70.093333 -1.420387
6             Tre Flowers  100           0.360000  0.030000     0.090580   75.586667 -2.126328
7            Bradley Roby   63           0.365079  0.031746     0.078652   85.100000 -2.334418
8           Ross Cockrell   67           0.417910  0.

In [23]:
# Cap int_rate at 0.05 to reduce the penalty for zero INTs
# This compresses the range so 0 INTs vs 5% INTs matters less
cb_season['int_rate_capped'] = cb_season['int_rate'].clip(upper=0.05)
labeled_season['int_rate_capped'] = labeled_season['int_rate'].clip(upper=0.05)
cb_2025['int_rate_capped'] = cb_2025['int_rate'].clip(upper=0.05)

features_capped = ['incompletion_rate', 'int_rate_capped', 'passer_rating_inv']

X = labeled_season[features_capped].values
y = labeled_season['relevance'].values
group = [len(X)]

train_data = lgb.Dataset(X, label=y, group=group)

params = {
    'objective':        'lambdarank',
    'metric':           'ndcg',
    'ndcg_eval_at':     [10],
    'learning_rate':    0.05,
    'num_leaves':       8,
    'min_data_in_leaf': 1,
    'verbose':          -1,
}

model_v4 = lgb.train(params, train_data, num_boost_round=200)

importance = model_v4.feature_importance(importance_type='gain')
weightages  = importance / importance.sum() * 100

print("Feature weightages:")
for feat, w in zip(features_capped, weightages):
    print(f"  {feat}: {w:.1f}%")

# Score 2025
X_2025 = cb_2025[features_capped].values
cb_2025['wcs_v4'] = model_v4.predict(X_2025)

cb_2025_ranked_v4 = cb_2025[['player', 'tgt', 'incompletion_rate', 'int_rate', 'target_rate', 'avg_rat', 'wcs_v4']]\
    .sort_values('wcs_v4', ascending=False)\
    .reset_index(drop=True)

cb_2025_ranked_v4.index += 1
print(f"\n2025 CB Rankings — Capped int rate:")
print(cb_2025_ranked_v4.head(30).to_string())

Feature weightages:
  incompletion_rate: 31.5%
  int_rate_capped: 24.9%
  passer_rating_inv: 43.6%

2025 CB Rankings — Capped int rate:
                player  tgt  incompletion_rate  int_rate  target_rate    avg_rat    wcs_v4
1           AJ Terrell   64           0.468750  0.000000     0.066806  76.980000  4.718006
2        Renardo Green   69           0.449275  0.000000     0.073482  80.107143  4.159403
3   Christian Gonzalez   84           0.464286  0.000000     0.082840  77.950000  3.772290
4       Mekhi Blackmon   76           0.342105  0.026316     0.097063  83.370588  3.015524
5    Tre'Davious White   46           0.456522  0.021739     0.055556  68.660000  2.250415
6        Darien Porter   35           0.342857  0.000000     0.057756  80.111111  1.976923
7    Ja'Quan McMillian   79           0.367089  0.025316     0.095642  75.500000  1.849402
8       Trent McDuffie   44           0.363636  0.022727     0.063953  81.781818  1.762466
9           Nate Hobbs   29           0.34482